<a href="https://colab.research.google.com/github/Samirarim/conference-paper-filter-2025/blob/main/CVPR_ICCV_ICML_2025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
from bs4 import BeautifulSoup
import csv

def fetch_cvf_papers(url, venue):
    # Extracts papers from CVPR and ICCV (Computer Vision Foundation)
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    papers = []

    for dt in soup.find_all('dt', class_='ptitle'):
        title = dt.text.strip()
        link_tag = dt.find('a')
        link = "https://openaccess.thecvf.com" + link_tag['href'] if link_tag else "N/A"
        papers.append({'Venue': venue, 'Title': title, 'Link': link})
    return papers

def fetch_pmlr_papers(url, venue):
    # Extracts papers from ICML (Proceedings of Machine Learning Research)
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    papers = []

    for p in soup.find_all('p', class_='title'):
        title = p.text.strip()
        link_tag = p.find('a')
        link = link_tag['href'] if link_tag else "N/A"
        papers.append({'Venue': venue, 'Title': title, 'Link': link})
    return papers

if __name__ == "__main__":
    print("Fetching CVPR 2025 papers...")
    cvpr = fetch_cvf_papers('https://openaccess.thecvf.com/CVPR2025?day=all', 'CVPR 2025')

    print("Fetching ICCV 2025 papers...")
    iccv = fetch_cvf_papers('https://openaccess.thecvf.com/ICCV2025?day=all', 'ICCV 2025')

    print("Fetching ICML 2025 papers...")
    icml = fetch_pmlr_papers('https://proceedings.mlr.press/v267/', 'ICML 2025')

    all_papers = cvpr + iccv + icml

    with open('2025_ai_vision_papers.csv', 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['Venue', 'Title', 'Link'])
        writer.writeheader()
        writer.writerows(all_papers)

    print(f"Successfully saved {len(all_papers)} papers to 2025_ai_vision_papers.csv")

Fetching CVPR 2025 papers...
Fetching ICCV 2025 papers...
Fetching ICML 2025 papers...
Successfully saved 8902 papers to 2025_ai_vision_papers.csv


In [ ]:
import pandas as pd

# 1. Load the CSV you just generated
df = pd.read_csv('2025_ai_vision_papers.csv')

# 2. Define target keywords for each domain
domain_keywords = {
    'Medical': ['medical', 'clinical', 'surgery', 'health', 'tumor', 'mri', 'radiology', '3d medical image', 'anatomy'],
    'Environment': ['environment', 'climate', 'weather', 'ecology', 'pollution', 'earth observation', 'satellite'],
    'Agri': ['agriculture', 'agricultural', 'crop', 'plant disease', 'leaf', 'farming', 'soil', 'harvest'],
    'Safety': ['safety', 'security', 'surveillance', 'hazard', 'collision', 'anomaly detection', 'autonomous driving'],
    'Multimodal': ['multimodal', 'multi-modal', 'vision-language', 'audio-visual', 'cross-modal', 'text-to-image']
}

# 3. Create a function to tag papers based on title keywords
def assign_domain(title):
    title_lower = str(title).lower()
    found_domains = []

    for domain, keywords in domain_keywords.items():
        if any(keyword in title_lower for keyword in keywords):
            found_domains.append(domain)

    # Join multiple domains if a paper overlaps (e.g., "Medical, Multimodal")
    return ", ".join(found_domains) if found_domains else "Uncategorized"

# 4. Apply the tagging and filter out the uncategorized papers
df['Matched_Domains'] = df['Title'].apply(assign_domain)
filtered_df = df[df['Matched_Domains'] != 'Uncategorized'].copy()

# 5. Save the final filtered dataset
filtered_df.to_csv('filtered_domain_papers.csv', index=False)

print(f"Filtering complete! Kept {len(filtered_df)} relevant papers out of {len(df)}.")
print("\nBreakdown of found domains:")
print(filtered_df['Matched_Domains'].value_counts().head(10))

Filtering complete! Kept 1139 relevant papers out of 8902.

Breakdown of found domains:
Matched_Domains
Multimodal                 768
Safety                     140
Medical                    101
Environment                 71
Medical, Multimodal         23
Safety, Multimodal          23
Agri                         6
Environment, Multimodal      4
Agri, Multimodal             2
Environment, Safety          1
Name: count, dtype: int64
